# 08 · Fine-Tuning an LLM with LoRA

**Time:** about 60 minutes (plus training) · **Prerequisites:** `07`

## Learning objectives
1. Explain LoRA: freeze a pretrained model and train small low-rank matrices
   added to its weights. With a 4-bit base model this is **QLoRA**.
2. Prepare chat-format data with **prompt masking**, so the loss covers only
   the answer.
3. **Measure the base model first**, on development data, then compare frozen choices
   on a final test with confidence intervals. A strong base model leaves little to gain.
4. Compare zero-shot, few-shot, and LoRA under the same output contract;
   measure format compliance and possible loss of unrelated capabilities.

### LoRA in one equation
A frozen weight matrix $W \in \mathbb{R}^{d \times k}$ gets a trainable update
of rank $r$:

$$ W' = W + \frac{\alpha}{r} BA, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d, k) $$

For $d = k = 2048$ (this model's hidden size) and $r = 8$, $BA$ has $2 \cdot 2048 \cdot 8 = 32{,}768$
trainable numbers instead of $4.2$ million. $B$ starts at zero, so training
begins exactly at the pretrained model. MLX-LM exposes the multiplier as
`scale` directly, so our config uses s=20 in W′=W+sBA; it does not silently
interpret 20 as alpha and divide it by rank.

**Model.** We use `mlx-community/Qwen2.5-3B-Instruct-4bit`, a 3-billion-parameter
instruction-tuned model quantized to 4 bits (about 1.7 GB to download on first
use). The exact snapshot is pinned in `config/models.json`. Install the `llm`
group and provide a model download budget (see the root README). To change
models, update the manifest revision and verify its architecture and memory needs.
Training takes several minutes on recent Apple Silicon; memory use depends on
batch size and sequence length.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import mlx.core as mx
from mlx_lm import generate, load
from mlx_lm.sample_utils import make_sampler

from mlx_nlp_utils import print_device_info
from experiment_utils import model_path, score_label, write_record, accuracy_interval

print_device_info()
MODEL_NAME = model_path("qwen_chat")  # immutable local snapshot; budget checked first
DATA_DIR = Path("../data").resolve()
ADAPTER_DIR = Path("adapters").resolve()          # notebooks/adapters
greedy = make_sampler(temp=0.0)                    # deterministic decoding for evaluation

## 1. Load the pretrained model

In [ ]:
model, tokenizer = load(MODEL_NAME)
messages = [{"role": "user", "content": "What is the capital of France?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
print(generate(model, tokenizer, prompt=prompt, max_tokens=40, sampler=greedy))

`apply_chat_template` wraps messages in the special tokens this model was
trained with. Always use the tokenizer's own template; hand-written
delimiters from a different model family silently hurt quality.

## 2. The task and the data

The task: reply with exactly one label (`greeting`, `question`, or
`command`) for a user message, the intent task from notebook 01, now done
by an LLM. `data/train.jsonl` and `data/valid.jsonl` hold chat records:

In [ ]:
train_records = [json.loads(line) for line in (DATA_DIR / "train.jsonl").read_text().splitlines()]
valid_records = [json.loads(line) for line in (DATA_DIR / "valid.jsonl").read_text().splitlines()]
print(json.dumps(train_records[0], indent=1))

unique = lambda records: sorted({r["messages"][1]["content"] for r in records})
print(f"\ntrain: {len(train_records)} records, {len(unique(train_records))} unique prompts")
print(f"valid: {len(valid_records)} records, {len(unique(valid_records))} unique prompts: {unique(valid_records)}")

Look at those counts. The 800 training records are only ~37 distinct
prompts repeated, and the 200 validation records are only 8. The generator
splits prompts *before* repeating them, so nothing leaks, but **record count
is not example count**. The validation file is fine for watching the loss
curve; it is far too small to measure accuracy.

So we write a separate **test set**: new, hand-written messages that
appear in neither file and use different wording from the templates.

In [ ]:
SYSTEM = "Classify the user message. Reply with exactly one label: greeting, question, or command."
LABELS = ["greeting", "question", "command"]

evaluation_pool = [
    ("Good evening, nice to see you", "greeting"), ("Hiya!", "greeting"), ("Morning team", "greeting"),
    ("Hey hey, long time no see", "greeting"), ("Greetings from Berlin", "greeting"),
    ("Hello again", "greeting"), ("Sup", "greeting"), ("Hi, hope you're well", "greeting"),
    ("Is the pharmacy open on Sundays?", "question"), ("How far is the airport?", "question"),
    ("What's the exchange rate for yen?", "question"), ("Who wrote Pride and Prejudice?", "question"),
    ("Why does my laptop get so hot?", "question"), ("When is the next train?", "question"),
    ("Which planet is the largest?", "question"), ("Do you know a good pizza place?", "question"),
    ("Dim the bedroom lights", "command"), ("Add milk to my shopping list", "command"),
    ("Start a five minute timer", "command"), ("Call the dentist", "command"),
    ("Turn the heating up two degrees", "command"), ("Text Sarah that I'm running late", "command"),
    ("Shuffle my workout playlist", "command"), ("Book a taxi for 7am", "command"),
]
seen = set(unique(train_records)) | set(unique(valid_records))
assert not any(text in seen for text, _ in evaluation_pool), "test messages must be new"
# Fixed balanced split: four examples per label for development, four for final test.
dev_set = [row for i, row in enumerate(evaluation_pool) if i % 8 < 4]
test_set = [row for i, row in enumerate(evaluation_pool) if i % 8 >= 4]
assert {x for x, _ in dev_set}.isdisjoint(x for x, _ in test_set)
print(f"Development: {len(dev_set)}; final test: {len(test_set)} (small illustrative sets)")

## 3. Set the output contract before comparing models

Use exactly one of `greeting`, `question`, or `command`. We report three
separate metrics: recoverable first-label accuracy, exact format compliance,
and strict accuracy (the whole normalized response equals the gold label).
`greeting because ...` can have the correct label while failing the contract.

All primary zero-shot, few-shot, and LoRA comparisons use the **same system
instruction**. Bare prompting is a separate ablation. Development examples
support choices; the final test is evaluated only after configuration selection.
The final test here is tiny, so it illustrates a protocol, not a benchmark.

We use a Wilson interval for binary accuracy. Unlike a naive bootstrap of
12/12 successes, it does not report [100%, 100%] certainty. The small sample
still gives a wide interval; perfect observed accuracy is not proof of perfection.


In [ ]:
FEW_SHOTS = [("Hello", "greeting"), ("What time is it?", "question"), ("Open the door", "command")]

def build_prompt(tokenizer, text, instructed=True, few_shot=False):
    messages = [{"role": "system", "content": SYSTEM}] if instructed else []
    if few_shot:
        for example, label in FEW_SHOTS:
            messages.extend([{"role": "user", "content": example}, {"role": "assistant", "content": label}])
    messages.append({"role": "user", "content": text})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def evaluate(model, tokenizer, examples, name, instructed=True, few_shot=False):
    rows = []
    for text, gold in examples:
        prompt = build_prompt(tokenizer, text, instructed, few_shot)
        reply = generate(model, tokenizer, prompt=prompt, max_tokens=32, sampler=greedy)
        rows.append({"text": text, "gold": gold, "reply": reply,
                     "prompt_tokens": len(tokenizer.encode(prompt)), **score_label(reply, gold, LABELS)})
    metrics = {key: float(np.mean([r[key] for r in rows]))
               for key in ["correct", "format_valid", "strict_correct"]}
    acc, low, high = accuracy_interval([r["strict_correct"] for r in rows])
    metrics.update(strict_ci=[low, high], mean_prompt_tokens=float(np.mean([r["prompt_tokens"] for r in rows])))
    print(name, metrics)
    return rows, metrics

base_dev, _ = evaluate(model, tokenizer, dev_set, "Zero-shot development")
few_dev, _ = evaluate(model, tokenizer, dev_set, "Few-shot development", few_shot=True)
bare_dev, _ = evaluate(model, tokenizer, dev_set, "Bare development ablation", instructed=False)


A six-token cap can hide format violations, so this comparison allows 32
output tokens and records raw replies. Inspect truncation and outputs as well
as aggregate metrics. Identical decoding and scoring rules apply to every method.

## 4. Train an adapter

Training keeps the same system instruction used in evaluation. Prompt masking
restricts loss to assistant answers; checkpointing and a 256-token sequence
limit reduce memory use. The pinned base model already has quantized weights.


In [ ]:
import os
import re

TRAIN_DATA = ADAPTER_DIR / "data_instructed"
TRAIN_DATA.mkdir(parents=True, exist_ok=True)
for name, records in [("train", train_records), ("valid", valid_records)]:
    with open(TRAIN_DATA / f"{name}.jsonl", "w") as f:
        for r in records:
            f.write(json.dumps({"messages": [{"role": "system", "content": SYSTEM}] + [m for m in r["messages"] if m["role"] != "system"]}) + "\n")

def train_adapter(adapter_dir, rank=8, iters=200, seed=0, scale=20.0):
    """Write a config file and run the mlx_lm LoRA trainer quietly. Returns (seconds, log text)."""
    adapter_dir.mkdir(parents=True, exist_ok=True)
    config = {
        "model": MODEL_NAME, "train": True, "data": str(TRAIN_DATA), "adapter_path": str(adapter_dir),
        "fine_tune_type": "lora", "num_layers": 16, "batch_size": 2, "iters": iters,
        "learning_rate": 1e-4, "max_seq_length": 256, "grad_checkpoint": True, "mask_prompt": True,
        "lora_parameters": {"rank": rank, "dropout": 0.0, "scale": scale},
        "steps_per_report": 20, "steps_per_eval": 50, "val_batches": 25, "seed": seed,
    }
    config_path = adapter_dir / "lora_config.json"
    config_path.write_text(json.dumps(config, indent=1))    # JSON is valid YAML
    start = time.perf_counter()
    result = subprocess.run([sys.executable, "-m", "mlx_lm", "lora", "-c", str(config_path)],
                            capture_output=True, text=True, env={**os.environ, "TOKENIZERS_PARALLELISM": "false",
                                                                "NO_COLOR": "1", "TERM": "dumb"})
    log = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", result.stdout + result.stderr).replace("\r", "\n")
    (adapter_dir / "train.log").write_text(log)
    if result.returncode != 0:
        raise RuntimeError("Training failed; last log lines:\n" + "\n".join(log.strip().splitlines()[-15:]))
    return time.perf_counter() - start, log

def trainable_count(adapter_dir):
    return sum(w.size for w in mx.load(str(adapter_dir / "adapters.safetensors")).values())

seconds, log = train_adapter(ADAPTER_DIR, rank=8, iters=200)
print(next((l.strip() for l in log.splitlines() if "Trainable parameters" in l), ""))
val_losses = re.findall(r"^\s*(\d+)\s+val\s+([\d.]+)", log, flags=re.M)
print("validation loss by iteration:", ", ".join(f"{i}: {v}" for i, v in val_losses))
print(f"\nTrained in {seconds / 60:.1f} min; adapter file holds {trainable_count(ADAPTER_DIR):,} numbers "
      f"(the base model has about 3 billion)")

Validation loss is measured on repeated synthetic prompts. It checks the
training loop, but does not establish generalization. Next select rank/scale
using development **strict accuracy**, without inspecting final test results.
The fixed default remains rank 8, scale 20 when the optional sweep is disabled.


## 5. Optional rank/scale selection on development data

MLX-LM uses an explicit update scale. Compare fixed scale with an alpha/r-style schedule; select by development strict accuracy. Ties keep the first configuration.


In [ ]:
RUN_RANK_SWEEP = False
selected_adapter = ADAPTER_DIR
best_dev_score = -1.0
rank_sweep_iters = 100

if RUN_RANK_SWEEP:
    sweep = []
    for rank in [2, 8, 32]:
        for mode, scale in [("fixed scale 20", 20.0), ("scale 20·8/rank", 20.0 * 8 / rank)]:
            path = ADAPTER_DIR / f"rank_{rank}_{'fixed' if scale == 20.0 else 'scaled'}"
            secs, log = train_adapter(path, rank=rank, iters=rank_sweep_iters, scale=scale)
            final_val = (re.findall(r"^\s*\d+\s+val\s+([\d.]+)", log, flags=re.M) or ["?"])[-1]
            m, t = load(MODEL_NAME, adapter_path=str(path))
            _, metrics = evaluate(m, t, dev_set, f"rank {rank}, {mode}")
            acc = metrics["strict_correct"]
            if acc > best_dev_score:
                best_dev_score, selected_adapter = acc, path
            sweep.append((rank, mode, scale, trainable_count(path), final_val, acc))
            del m
    print(f"\n{'rank':>4} {'scale':>6} {'mode':<17} {'trainable':>11} {'final val loss':>15} {'dev strict':>9}")
    for rank, mode, scale, params, val, acc in sweep:
        print(f"{rank:>4} {scale:>6g} {mode:<17} {params:>11,} {val:>15} {acc:>9.0%}")
else:
    print("Skipped. Set RUN_RANK_SWEEP = True to train and compare ranks (about ten minutes).")

## 6. Freeze choices, then evaluate the final test

Do not use this table to retune and then report the same test as untouched. The three unrelated probes illustrate potential capability loss; they are not a comprehensive forgetting benchmark.


In [ ]:
ft_model, ft_tokenizer = load(MODEL_NAME, adapter_path=str(selected_adapter))
final_results = {}
for name, m, t, few in [("zero_shot", model, tokenizer, False),
                         ("few_shot", model, tokenizer, True),
                         ("lora", ft_model, ft_tokenizer, False)]:
    rows, metrics = evaluate(m, t, test_set, name, few_shot=few)
    final_results[name] = {"metrics": metrics, "predictions": rows}

# A small held-out general-capability probe; no classification system message.
general_probes = [("What is 2 + 2? Reply with the number only.", "4"),
                  ("What is the capital of France? Reply with the city only.", "paris"),
                  ("Translate cat into Spanish. Reply with one word.", "gato")]
for name, m, t in [("base", model, tokenizer), ("lora", ft_model, ft_tokenizer)]:
    replies = [generate(m, t, prompt=build_prompt(t, text, instructed=False),
                        max_tokens=32, sampler=greedy).strip().lower() for text, _ in general_probes]
    final_results[name + "_general"] = {"replies": replies,
        "exact_accuracy": float(np.mean([r == gold for r, (_, gold) in zip(replies, general_probes)]))}
print(write_record("08-finetuning", models=["qwen_chat"], seed=0,
    settings={"adapter": str(selected_adapter), "max_tokens": 32, "temperature": 0,
              "system": SYSTEM, "dev_set": dev_set, "test_set": test_set, "few_shots": FEW_SHOTS},
    metrics=final_results, data_files=[DATA_DIR / "train.jsonl", DATA_DIR / "valid.jsonl",
                                    selected_adapter / "lora_config.json", selected_adapter / "adapters.safetensors"]))


## Exercises and interpretation

1. Why can label accuracy improve while format compliance falls? Inspect raw replies.
2. Compare zero-shot, few-shot, and LoRA strict accuracy and prompt token counts. Is training worth it here?
3. Expand development and final sets with disjoint paraphrase families before comparing ranks.
4. Expand unrelated probes before claiming capability preservation.

The old numerical tables used different prompts, permissive scoring, and test-set sweeps; they do not apply to this protocol. Regenerate results with the pinned snapshot. See [MLX-LM LoRA documentation](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md).
